# LaCAM centralized baseline (reviewer S3) — Colab/Kaggle

Runs the **real LaCAM** solver (Kei18/lacam3, via the official
`Cognitive-AI-Systems/pogema-benchmark` integration) on our den520d saturation
densities, so we have a centralized planner for context. CPU-only; no GPU needed.

**Order:** run cells top to bottom. The C++ build (cell 3) is the step most likely
to need attention — confirm `liblacam.so` exists before moving on. The **smoke test**
(cell 6) verifies LaCAM actually runs one episode before the full sweep (cell 7).

Why a separate notebook: pogema-benchmark's `eval.py` imports *every* solver
(RHCR/SCRIMP/DCC/…), each needing its own build. We import **only LaCAM** + their
`create_env`, and run it through the same `pogema_toolbox.evaluation()` we use for
Follower — so the throughput numbers are directly comparable.

## 1. System build tools + clone repos

In [ ]:
# CMake/compiler for the LaCAM C++ build, then clone pogema-benchmark RECURSIVELY
# (its lacam3 submodule is the actual solver) plus our repo (for the den520d map).
!apt-get -qq update && apt-get -qq install -y cmake build-essential >/dev/null
import os
if not os.path.exists('/content/pogema-benchmark'):
    !git clone --recursive -q https://github.com/Cognitive-AI-Systems/pogema-benchmark.git /content/pogema-benchmark
# make sure the lacam3 submodule actually came down
!cd /content/pogema-benchmark && git submodule update --init --recursive
if not os.path.exists('/content/mapf-deadlock'):
    !git clone -q --depth 1 https://github.com/tay805/mapf-deadlock.git /content/mapf-deadlock
print('lacam3 present:', os.path.isdir('/content/pogema-benchmark/algorithms/lacam/lacam3'))

## 2. Build `liblacam.so`

In [ ]:
# Build the LaCAM shared library and place it where inference.py expects it
# (algorithms/lacam/liblacam.so; default lacam_lib_path = 'lacam/liblacam.so').
%cd /content/pogema-benchmark/algorithms/lacam
!cmake -B build -DCMAKE_BUILD_TYPE=Release >/tmp/cmake.log 2>&1 && cmake --build build -j 2 >>/tmp/cmake.log 2>&1; tail -5 /tmp/cmake.log
import glob, shutil, os
so = glob.glob('/content/pogema-benchmark/algorithms/lacam/**/liblacam.so', recursive=True)
print('built:', so)
assert so, 'liblacam.so not built — see /tmp/cmake.log'
shutil.copy(so[0], '/content/pogema-benchmark/algorithms/lacam/liblacam.so')
print('placed at algorithms/lacam/liblacam.so')

## 3. Python env (py3.10 conda) + LaCAM's deps
Same py3.10 `follower` conda env strategy as `colab_baseline.ipynb` (Colab's native
3.12 breaks the pinned MAPF stack). If you already built that env this session, this
reuses it. Kaggle note: Kaggle is already py3.10, so there you can skip condacolab and
`pip install` the packages directly into the kernel.

In [ ]:
import os, subprocess, sys
USE_CONDA = True   # Colab: True (build py3.10 env). Kaggle py3.10: set False to pip-install into kernel.
PKGS = 'pogema==1.3.0 pogema-toolbox==0.1.0 numpy<=1.23.1 pandas<=1.4 pyyaml dask==2024.8.0 distributed==2024.8.0 loguru cppimport pybind11==2.13.1 tabulate'
if USE_CONDA:
    try:
        import condacolab; condacolab.check()
    except Exception:
        !pip install -q condacolab
        import condacolab; condacolab.install()   # kernel restarts here; re-run this cell after
    # create env once
    envs = subprocess.run(['conda','env','list'],capture_output=True,text=True).stdout
    if 'follower' not in envs:
        !conda create -y -n follower python=3.10 >/dev/null
    !conda run -n follower pip install -q --prefer-binary {PKGS}
    print('conda env follower ready')
else:
    !pip install -q --prefer-binary {PKGS}
    print('kernel deps installed')

## 4. Our scenario config + map + minimal LaCAM runner

In [ ]:
# Write a den520d saturation config (our densities), its maps.yaml (den520d pulled
# from our repo), and a LaCAM-ONLY runner modelled on pogema-benchmark/eval.py.
import yaml, os
EXP = '/content/pogema-benchmark/algorithms/experiments/99-den520d-sat'
os.makedirs(EXP, exist_ok=True)
# den520d map from our repo, in pogema-toolbox map format
ours = yaml.safe_load(open('/content/mapf-deadlock/learn-to-follow/env/test-maps.yaml'))
yaml.safe_dump({'den520d': ours['den520d']}, open(f'{EXP}/maps.yaml','w'))
cfg = {
    'environment': {
        'name': 'Environment', 'collision_system': 'soft', 'on_target': 'restart',
        'observation_type': 'POMAPF', 'max_episode_steps': 512, 'map_name': 'den520d',
        'num_agents': {'grid_search': [128, 256, 384, 512, 640]},
        'seed': {'grid_search': [0, 1, 2]},
    },
    'algorithms': {'LaCAM': {'name': 'LaCAM'}},
}
yaml.safe_dump(cfg, open(f'{EXP}/99-den520d-sat-lmapf.yaml','w'))
runner = '''import sys; sys.argv=["r"]
from pathlib import Path
import yaml
from pogema_toolbox.evaluator import evaluation
from pogema_toolbox.registry import ToolboxRegistry
from create_env import create_env_base
from pogema_toolbox.create_env import Environment
from lacam.inference import LacamInference, LacamInferenceConfig
ToolboxRegistry.register_env("Environment", create_env_base, Environment)
ToolboxRegistry.register_algorithm("LaCAM", LacamInference, LacamInferenceConfig)
folder = Path("experiments/99-den520d-sat")
ToolboxRegistry.register_maps(yaml.safe_load(open(folder/"maps.yaml")))
cfg = yaml.safe_load(open(folder/"99-den520d-sat-lmapf.yaml"))
if len(sys.argv) > 1: pass
evaluation(cfg, eval_dir=folder)
print("DONE")
'''
open('/content/pogema-benchmark/algorithms/run_lacam.py','w').write(runner)
print('wrote config, maps.yaml, run_lacam.py')

## 5. Smoke test — 1 seed, 128 agents (verifies the .so loads & LaCAM runs)

In [ ]:
# Patch the config to a single quick instance, run, then restore the full grid.
import yaml
p = f'{EXP}/99-den520d-sat-lmapf.yaml'
full = yaml.safe_load(open(p))
quick = yaml.safe_load(open(p))
quick['environment']['num_agents'] = {'grid_search': [128]}
quick['environment']['seed'] = {'grid_search': [0]}
quick['environment']['max_episode_steps'] = 128
yaml.safe_dump(quick, open(p,'w'))
%cd /content/pogema-benchmark/algorithms
RUN = 'conda run --no-capture-output -n follower python' if True else 'python'
!{RUN} run_lacam.py
import json, glob
j = glob.glob('experiments/99-den520d-sat/*.json')
print('result files:', j)
if j:
    r = json.load(open(j[0]))
    print('smoke throughput:', r[0]['metrics'].get('avg_throughput'))
yaml.safe_dump(full, open(p,'w'))   # restore full grid for the sweep
print('restored full config')

## 6. Full sweep — den520d 128–640, 3 seeds (the S3 result)

In [ ]:
# ~15 episodes; LaCAM replans on goal changes (lifelong). Saves throughput per
# (agents, seed). Copy the JSON back to share, or read it in the next cell.
%cd /content/pogema-benchmark/algorithms
!conda run --no-capture-output -n follower python run_lacam.py
!cp experiments/99-den520d-sat/*.json /content/ 2>/dev/null; echo saved

## 7. Throughput vs density (compare to Follower / density control)

In [ ]:
import json, glob
from collections import defaultdict
j = glob.glob('/content/pogema-benchmark/algorithms/experiments/99-den520d-sat/*.json')
agg = defaultdict(list)
for r in json.load(open(j[0])):
    agg[r['env_grid_search']['num_agents']].append(r['metrics']['avg_throughput'])
ref = {128:2.23,256:2.32,384:1.92,512:1.56,640:1.06}   # Follower all-active (3 seeds)
print(f"{'agents':>6}{'LaCAM tp':>10}{'Follower':>10}")
for n in sorted(agg):
    t = sum(agg[n])/len(agg[n])
    print(f"{n:>6}{t:>10.3f}{ref.get(n,0):>10.2f}")
print('\nDensity control (den520d, hide-meter): 192 active -> ~2.47 even with 640 present.')